README!

The following directories will be setup in the directory where code executes:<br>
a. papers where all pdf papers for summarization and Q/A is used<br>
b. results where all results and metrics will be stored<br>
c. reference where the json files for the "golden" answer for Question-Answer and summarization is stored.<br>
d. models where the LLM models for evaluation and later if used, the model for judging will be stored.<br>
e. code assumes your secret keys are stored in an .env file. The default .env file name is "AI6130.env".  if not found it will ask for you to manually enter the secret keys

In [ ]:
#Run this once in local or on colab
!huggingface-cli download meta-llama/Llama-3.1-8B --local-dir Llama-3.1-8B # remove if already downloaded the model from HuggingFace
!huggingface-cli download deepseek-ai/DeepSeek-R1-Distill-Llama-8B --local-dir DeepSeek-R1-Distill-Llama-8B
!pip install -U langchain langchain-huggingface transformers sentence-transformers
!pip install langchain-community
!pip install transformers
!pip install python-dotenv
!pip install huggingface_hub
!pip install langchain-huggingface
!pip install ragas langchain transformers huggingface_hub pandas torch
!pip install --upgrade ragas
!pip install bert-score rouge nltk evaluate datasets
!pip install accelerate bitsandbytes
!pip install pypdf

In [1]:
import os
import sys
import torch
import gc
import logging
import random
import json
import pandas as pd
import numpy as np
import re
import time
import warnings
from typing import List, Dict, Any, Tuple
from pathlib import Path
from dotenv import load_dotenv
from glob import glob
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# New imports for evaluation metrics
import evaluate
from rouge import Rouge
from bert_score import score as bert_score
import nltk
from nltk.tokenize import word_tokenize

# For RAGAS evaluation
from ragas.metrics import (
    context_precision, 
    context_recall, 
    faithfulness, 
    answer_relevancy,
    answer_correctness
)
from ragas import evaluate as ragas_evaluate
from datasets import Dataset

# Optional for open LLM judge
from transformers import pipeline

# Download NLTK data
nltk.download('punkt', quiet=True)


True

In [2]:
#Load data here
def load_document(file_path):
    logger.info(f"Loading document: {file_path}")
    try:
        if file_path.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
            return loader.load()
        elif file_path.endswith(('.txt', '.md')):
            loader = TextLoader(file_path)
            return loader.load()
        else:
            logger.error(f"Unsupported file type: {file_path}")
            return []
    except Exception as e:
        logger.error(f"Error loading file {file_path}: {str(e)}")
        return []

def load_pdf_documents(papers_dir: str) -> Dict[str, str]:
    """
    Load multiple PDF documents from papers directory
    Returns a dictionary mapping filename to content
    """
    paper_files = [f for f in os.listdir(papers_dir) if f.endswith('.pdf')]
    if not paper_files:
        logger.error(f"No PDF files found in {papers_dir}")
        return {}
    
    paper_contents = {}
    for paper_file in paper_files:
        paper_path = os.path.join(papers_dir, paper_file)
        logger.info(f"Loading document: {paper_path}")
        try:
            doc_sections = load_document(paper_path)
            if doc_sections:
                content = "\n\n".join([section.page_content for section in doc_sections])
                paper_contents[paper_file] = content
                logger.info(f"Successfully loaded {paper_file} ({len(content)} characters)")
            else:
                logger.warning(f"No content loaded from {paper_file}")
        except Exception as e:
            logger.error(f"Error loading file {paper_file}: {str(e)}")
    
    return paper_contents

def load_reference_data(reference_dir: str) -> Tuple[Dict[str, List[Dict]], Dict[str, str]]:
    """
    Load reference QA pairs and summaries from the reference directory
    Returns a tuple of (qa_pairs_dict, summaries_dict)
    """
    qa_pairs_dict = {}
    summaries_dict = {}
    
    # Load QA pairs
    qa_file = os.path.join(reference_dir, "qa_pairs.json")
    if os.path.exists(qa_file):
        try:
            with open(qa_file, "r") as f:
                qa_pairs_dict = json.load(f)
            logger.info(f"Loaded QA pairs for {len(qa_pairs_dict)} papers")
        except Exception as e:
            logger.error(f"Error loading QA pairs: {str(e)}")
    else:
        logger.warning(f"QA pairs file not found at {qa_file}")
    
    # Load summaries
    summary_files = glob(os.path.join(reference_dir, "*_summary.json"))
    for summary_file in summary_files:
        try:
            paper_name = os.path.basename(summary_file).replace("_summary.json", "")
            with open(summary_file, "r") as f:
                summary_data = json.load(f)
            summaries_dict[paper_name] = summary_data.get("summary", "")
            logger.info(f"Loaded summary for {paper_name}")
        except Exception as e:
            logger.error(f"Error loading summary from {summary_file}: {str(e)}")
    
    return qa_pairs_dict, summaries_dict

In [3]:
#Generate answer from LLM
def generate_answer(model, tokenizer, paper_content, question, device, seed=None, model_name=""):
    """
    Generate an answer to a question based on paper content
    """
    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that answers questions based on academic papers. 
You should provide concise, accurate answers based solely on the information in the paper context provided."""

    # Create user prompt with the paper content as context
    user_prompt = f"""Based on the following academic paper, please answer the question:

PAPER CONTENT:
{paper_content[:3000]}  # Limiting context size to avoid token limits

QUESTION:
{question}

Please provide a direct, concise answer using only information from the paper. If the paper doesn't contain relevant information to answer the question, please state that."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger.info(f"Generating answer using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,  # Shorter for answers
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

def generate_summary(model, tokenizer, paper_content, device, seed=None, model_name=""):
    """
    Generate a summary of a paper
    """
    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that summarizes academic papers. 
You should provide comprehensive, accurate summaries that capture the key points, methodologies, and findings of the paper."""

    # Create user prompt with the paper content as context
    user_prompt = f"""Please summarize the following documents:

PAPER CONTENT:
{paper_content[:3000]}  # Limiting context size to avoid token limits

Your summary should include:
1. The key concepts presented
2. Any practical applications or real-world examples
3. Include any historical context or notable scientists
4. The significance and implications of the concepts

Please be comprehensive yet concise and keep the summary to 300 words."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger.info(f"Generating summary using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1000,  # Longer for summaries
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

In [4]:
#Evaluate functions here
def evaluate_with_bertscore(generated_texts, reference_texts):
    """
    Evaluate generated texts using BERTScore
    """
    try:
        precision, recall, f1 = bert_score(
            generated_texts, 
            reference_texts, 
            lang="en", 
            verbose=True
        )
        return {
            "bertscore_precision": precision.mean().item(),
            "bertscore_recall": recall.mean().item(),
            "bertscore_f1": f1.mean().item()
        }
    except Exception as e:
        logger.error(f"Error calculating BERTScore: {str(e)}")
        return {
            "bertscore_precision": 0,
            "bertscore_recall": 0,
            "bertscore_f1": 0
        }

def evaluate_with_rouge(generated_texts, reference_texts):
    """
    Evaluate generated texts using ROUGE
    """
    try:
        rouge = Rouge()
        scores = rouge.get_scores(generated_texts, reference_texts, avg=True)
        return {
            "rouge1_f": scores["rouge-1"]["f"],
            "rouge2_f": scores["rouge-2"]["f"],
            "rougeL_f": scores["rouge-l"]["f"]
        }
    except Exception as e:
        logger.error(f"Error calculating ROUGE: {str(e)}")
        return {
            "rouge1_f": 0,
            "rouge2_f": 0,
            "rougeL_f": 0
        }

def setup_ragas_evaluation(use_open_llm=False, open_llm_name=None, openai_key=None, hf_token=None):
    """
    Setup RAGAS evaluation metrics and LLM
    """
    try:
        # Initialize metrics
        ragas_metrics = [
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
            answer_correctness
        ]
        
        if use_open_llm and open_llm_name:
            # Use open LLM via Hugging Face API
            if hf_token:
                from langchain_huggingface import HuggingFaceEndpoint
                
                # Create LangChain wrapper for the Hugging Face API endpoint
                ragas_llm = HuggingFaceEndpoint(
                    endpoint_url=f"https://api-inference.huggingface.co/models/{open_llm_name}",
                    huggingfacehub_api_token=hf_token,
                    task="text-generation",
                    max_new_tokens=512,
                    temperature=0.1,
                    model_kwargs={
                        "do_sample": False
                    }
                )
                logger.info(f"Using {open_llm_name} via Hugging Face API as judge")
            else:
                logger.warning("Hugging Face token not provided. Falling back to local model.")
                # Use local model pipeline
                from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
                import torch
                from transformers import pipeline

                pipe = pipeline(
                    "text-generation",
                    model=open_llm_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    max_new_tokens=512,
                    do_sample=False
                )
                
                ragas_llm = HuggingFacePipeline(pipeline=pipe)
                logger.info(f"Using local model as judge: {open_llm_name}")
        else:
            # Use OpenAI model
            if openai_key:
                from langchain_openai import ChatOpenAI
                # Explicitly set the API key
                import os
                os.environ["OPENAI_API_KEY"] = openai_key
                ragas_llm = ChatOpenAI(model_name="gpt-3.5-turbo", openai_api_key=openai_key)
                logger.info("Using OpenAI GPT-3.5-Turbo as judge")
            else:
                logger.warning("OpenAI API key not provided. RAGAS evaluation will not be available.")
                return None, None
        
        return ragas_metrics, ragas_llm
    except Exception as e:
        logger.error(f"Error setting up RAGAS evaluation: {str(e)}")
        return None, None

In [5]:
#Run RAGAS here
def run_ragas_for_qa(paper_content, questions, generated_answers, reference_answers, ragas_metrics, ragas_llm):
    """
    Run RAGAS evaluation for QA task
    """
    try:
        # Prepare data for RAGAS
        # For Q&A, we need contexts, questions, answers, and references
        contexts = [[paper_content]] * len(questions)  # Wrap each in a list as RAGAS expects list of contexts
        
        eval_data = {
            "contexts": contexts,
            "question": questions,
            "answer": generated_answers,
            "reference": reference_answers
        }
        
        # Create Dataset
        dataset = Dataset.from_dict(eval_data)
        
        # Run evaluation
        result = ragas_evaluate(
            dataset=dataset,
            metrics=ragas_metrics,
            llm=ragas_llm
        )
        
        return result
    except Exception as e:
        logger.error(f"Error in RAGAS QA evaluation: {str(e)}")
        return None

def run_ragas_for_summary(paper_content, generated_summary, reference_summary, ragas_metrics, ragas_llm):
    """
    Run RAGAS evaluation for summary task
    """
    try:
        # For summaries, we create a dummy question asking for a summary
        question = ["Summarize the key points of this academic paper"]
        
        # Prepare data for RAGAS
        eval_data = {
            "contexts": [[paper_content]],  # Wrap in a list as RAGAS expects list of contexts
            "question": question,
            "answer": [generated_summary],
            "reference": [reference_summary]
        }
        
        # Create Dataset
        dataset = Dataset.from_dict(eval_data)
        
        # Run evaluation
        result = ragas_evaluate(
            dataset=dataset,
            metrics=ragas_metrics,
            llm=ragas_llm
        )
        
        return result
    except Exception as e:
        logger.error(f"Error in RAGAS summary evaluation: {str(e)}")
        return None

In [6]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("paper_lesson_generator.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Function to clear GPU/MPS memory
def clear_memory():
    gc.collect()
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()
    return True

In [16]:
# Helper class to manage multiple models
class ModelManager:
    def __init__(self, model_configs, use_gpu=True):
        """
        Initialize with multiple model configurations
        
        Args:
            model_configs: List of dictionaries with model_path and model_name keys
            use_gpu: Whether to use MPS or CUDA for acceleration
        """
        self.model_configs = model_configs
        self.use_gpu = use_gpu
        self.cuda_available = torch.cuda.is_available()
        self.mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

        if use_gpu:
            if torch.cuda.is_available():
                self.device = "cuda"
            elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
                self.device = "mps"
        else:
            self.device = "cpu"
        print(f"Using device: {use_gpu}")
        # Will hold loaded models and tokenizers
        self.models = {}
        self.tokenizers = {}
        
        logger.info(f"ModelManager initialized with device: {self.device}")
        logger.info(f"Models to load: {[cfg['model_name'] for cfg in model_configs]}")
    
    def load_model(self, model_key):
        """Load a specific model by its key in model_configs"""
        if model_key in self.models and model_key in self.tokenizers:
            logger.info(f"Model {model_key} already loaded")
            return self.models[model_key], self.tokenizers[model_key]
        
        # Find model config
        config = next((cfg for cfg in self.model_configs if cfg['model_name'] == model_key), None)
        if not config:
            raise ValueError(f"Model {model_key} not found in configurations")
        
        model_path = config['model_path']
        logger.info(f"Loading model: {model_key} from {model_path}")
        
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_path, 
            use_fast=True, 
            local_files_only=True
        )
        
        # Load model
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            load_in_8bit=True if self.device == "cuda" else False, 
            torch_dtype=torch.float16 if self.device in ["cuda", "mps"] else torch.float32,
            low_cpu_mem_usage=True,
            local_files_only=True
        )
        
        # Move model to device
        model.to(self.device)
        
        # Store loaded model and tokenizer
        self.models[model_key] = model
        self.tokenizers[model_key] = tokenizer
        
        logger.info(f"Model {model_key} loaded successfully")
        return model, tokenizer
    
    def unload_model(self, model_key):
        """Unload a model to free up memory"""
        if model_key in self.models:
            logger.info(f"Unloading model: {model_key}")
            del self.models[model_key]
            if model_key in self.tokenizers:
                del self.tokenizers[model_key]
            clear_memory()
            return True
        return False
    
    def unload_all_models(self):
        """Unload all models"""
        logger.info("Unloading all models")
        self.models.clear()
        self.tokenizers.clear()
        clear_memory()
        return True
    
    def get_model_names(self):
        """Return list of available model names"""
        return [cfg['model_name'] for cfg in self.model_configs]


In [13]:
def check_and_setup_api_keys():
    #Check if API keys are set and prompt user if not available"
    if os.getenv("COLAB_RELEASE_TAG"):
        from google.colab import userdata
        openai_key = os.environ.get("OPENAI_API_KEY")
        hf_token = os.environ.get("HUGGINGFACE_API_TOKEN")
    else:
        env_path = Path("C:/Users/luqma/AI6130/AI6130_Grp/AI6130.env")
        if env_path.exists():
            load_dotenv(env_path)
            openai_key = os.environ.get("OPENAI_API_KEY")
            print("OpenAI API key found in environment variables.")
            hf_token = os.environ.get("HUGGINGFACE_API_TOKEN")
            print("Hugging Face API token found in environment variables.")
            return openai_key, hf_token
        else:
            print("OpenAI API key not found in environment variables.")
            use_openai = input("Do you want to use OpenAI models for evaluation? (y/n): ").lower() == 'y'
            
            if use_openai:
                openai_key = input("Please enter your OpenAI API key: ")
                os.environ["OPENAI_API_KEY"] = openai_key
                print("OpenAI API key set for this session.")
            else:
                print("OpenAI evaluation will not be available.")
        
            print("Hugging Face API token not found in environment variables.")
            use_hf_api = input("Do you want to use Hugging Face API for evaluation? (y/n): ").lower() == 'y'

            if use_hf_api:
                hf_token = input("Please enter your Hugging Face API token: ")
                os.environ["HUGGINGFACE_API_TOKEN"] = hf_token
                print("Hugging Face API token set for this session.")
            else:
                print("Hugging Face API evaluation will not be available.")
    
    return openai_key , hf_token

In [10]:
def run_experiment(model_configs, papers_dir="./papers", reference_dir="./reference", 
                  output_dir="./results", use_gpu=True, use_open_llm=False,
                  open_llm_name=None, openai_key=None, hf_token=None):
    """
    Run the full experiment with multiple PDFs, models, and evaluation metrics
    """
    papers_dir = Path(papers_dir)
    reference_dir = Path(reference_dir)
    output_dir = Path(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup logging
    experiment_log_path = os.path.join(output_dir, "experiment.log")
    file_handler = logging.FileHandler(experiment_log_path)
    file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
    logger.addHandler(file_handler)
    
    device = "cpu"
    if use_gpu:
        if torch.cuda.is_available():
            device = "cuda"
            logger.info(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
            logger.info("Using MPS (Metal Performance Shaders)")
        else:
            logger.info("No GPU available, using CPU")
    
    # Initialize model manager
    model_manager = ModelManager(model_configs, use_gpu=use_gpu)
    model_names = model_manager.get_model_names()
    
    # Load papers
    paper_contents = load_pdf_documents(papers_dir)
    if not paper_contents:
        logger.error("No paper contents loaded. Aborting experiment.")
        return
    
    # Load reference data
    qa_pairs_dict, reference_summaries = load_reference_data(reference_dir)
    
    # Setup RAGAS evaluation
    ragas_metrics, ragas_llm = setup_ragas_evaluation(use_open_llm, open_llm_name, openai_key, hf_token)
    if not ragas_metrics or not ragas_llm:
        logger.error("Failed to set up RAGAS evaluation. Continuing without RAGAS.")
    
    # Store all results
    all_results = {
        "qa_results": {},
        "summary_results": {},
        "bertscore_results": {},
        "rouge_results": {},
        "ragas_qa_results": {},
        "ragas_summary_results": {}
    }
    
    # Process each paper with each model
    for paper_file, paper_content in paper_contents.items():
        paper_basename = os.path.splitext(paper_file)[0]
        logger.info(f"Processing paper: {paper_file}")
        
        # Get QA pairs for this paper
        qa_pairs = qa_pairs_dict.get(paper_basename, [])
        if not qa_pairs:
            logger.warning(f"No QA pairs found for {paper_basename}. Skipping QA task.")
        
        # Get reference summary for this paper
        reference_summary = reference_summaries.get(paper_basename, "")
        if not reference_summary:
            logger.warning(f"No reference summary found for {paper_basename}. Skipping summary evaluation.")
        
        for model_name in model_names:
            logger.info(f"Processing with model: {model_name}")
            
            try:
                # Load model
                model, tokenizer = model_manager.load_model(model_name)
                
                # Generate answers to questions
                if qa_pairs:
                    questions = [pair["question"] for pair in qa_pairs]
                    reference_answers = [pair["answer"] for pair in qa_pairs]
                    
                    generated_answers = []
                    for i, question in enumerate(questions):
                        answer = generate_answer(
                            model, tokenizer, paper_content, question, 
                            device, seed=hash(paper_file + question) % 10000, 
                            model_name=model_name
                        )
                        generated_answers.append(answer)
                        
                        # Save generated answer
                        qa_output_dir = os.path.join(output_dir, "qa", paper_basename, model_name)
                        os.makedirs(qa_output_dir, exist_ok=True)
                        with open(os.path.join(qa_output_dir, f"question_{i+1}.txt"), "w") as f:
                            f.write(f"QUESTION:\n{question}\n\nANSWER:\n{answer}")
                    
                    # Store QA results
                    all_results["qa_results"][(paper_basename, model_name)] = {
                        "questions": questions,
                        "generated_answers": generated_answers,
                        "reference_answers": reference_answers
                    }
                    
                    # Evaluate QA with BERTScore and ROUGE
                    bertscore_results = evaluate_with_bertscore(generated_answers, reference_answers)
                    rouge_results = evaluate_with_rouge(generated_answers, reference_answers)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "qa")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "qa")] = rouge_results
                    
                    # Run RAGAS for QA if available
                    if ragas_metrics and ragas_llm:
                        ragas_qa_result = run_ragas_for_qa(
                            paper_content, questions, generated_answers, 
                            reference_answers, ragas_metrics, ragas_llm
                        )
                        if ragas_qa_result:
                            all_results["ragas_qa_results"][(paper_basename, model_name)] = ragas_qa_result
                
                # Generate summary
                generated_summary = generate_summary(
                    model, tokenizer, paper_content, device, 
                    seed=hash(paper_file) % 10000, model_name=model_name
                )
                
                # Save generated summary
                summary_output_dir = os.path.join(output_dir, "summaries", paper_basename)
                os.makedirs(summary_output_dir, exist_ok=True)
                with open(os.path.join(summary_output_dir, f"{model_name}_summary.txt"), "w") as f:
                    f.write(generated_summary)
                
                # Store summary results
                all_results["summary_results"][(paper_basename, model_name)] = {
                    "generated_summary": generated_summary,
                    "reference_summary": reference_summary
                }
                
                # Evaluate summary with BERTScore and ROUGE if reference exists
                if reference_summary:
                    bertscore_results = evaluate_with_bertscore([generated_summary], [reference_summary])
                    rouge_results = evaluate_with_rouge(generated_summary, reference_summary)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "summary")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "summary")] = rouge_results
                    
                    # Run RAGAS for summary if available
                    if ragas_metrics and ragas_llm:
                        ragas_summary_result = run_ragas_for_summary(
                            paper_content, generated_summary, reference_summary, 
                            ragas_metrics, ragas_llm
                        )
                        if ragas_summary_result:
                            all_results["ragas_summary_results"][(paper_basename, model_name)] = ragas_summary_result
                
                # Unload model to free memory
                model_manager.unload_model(model_name)
                clear_memory()
                
            except Exception as e:
                logger.error(f"Error processing {paper_file} with model {model_name}: {str(e)}")
    
    # Save all results to CSV files
    save_results_to_csv(all_results, output_dir)
    
    # Generate summary report
    generate_report(all_results, output_dir)
    
    logger.info("Experiment completed successfully!")
    return all_results

In [11]:
def save_results_to_csv(all_results, output_dir):
    """
    Save experiment results to CSV files
    """
    # Save BERTScore results
    bertscore_data = []
    for (paper, model, task), scores in all_results["bertscore_results"].items():
        row = {
            "paper": paper,
            "model": model,
            "task": task,
            **scores
        }
        bertscore_data.append(row)
    
    if bertscore_data:
        bertscore_df = pd.DataFrame(bertscore_data)
        bertscore_df.to_csv(os.path.join(output_dir, "bertscore_results.csv"), index=False)
    
    # Save ROUGE results
    rouge_data = []
    for (paper, model, task), scores in all_results["rouge_results"].items():
        row = {
            "paper": paper,
            "model": model,
            "task": task,
            **scores
        }
        rouge_data.append(row)
    
    if rouge_data:
        rouge_df = pd.DataFrame(rouge_data)
        rouge_df.to_csv(os.path.join(output_dir, "rouge_results.csv"), index=False)
    
    # Save RAGAS QA results
    if all_results["ragas_qa_results"]:
        ragas_qa_dfs = []
        for (paper, model), result in all_results["ragas_qa_results"].items():
            df = result.to_pandas()
            df["paper"] = paper
            df["model"] = model
            ragas_qa_dfs.append(df)
        
        ragas_qa_df = pd.concat(ragas_qa_dfs, ignore_index=True)
        ragas_qa_df.to_csv(os.path.join(output_dir, "ragas_qa_results.csv"), index=False)
    
    # Save RAGAS summary results
    if all_results["ragas_summary_results"]:
        ragas_summary_dfs = []
        for (paper, model), result in all_results["ragas_summary_results"].items():
            df = result.to_pandas()
            df["paper"] = paper
            df["model"] = model
            ragas_summary_dfs.append(df)
        
        ragas_summary_df = pd.concat(ragas_summary_dfs, ignore_index=True)
        ragas_summary_df.to_csv(os.path.join(output_dir, "ragas_summary_results.csv"), index=False)

def generate_report(all_results, output_dir):
    """
    Generate a markdown report summarizing the experiment results
    """
    report_path = os.path.join(output_dir, "experiment_report.md")
    
    with open(report_path, "w") as f:
        f.write("# RAG Experiment Report\n\n")
        f.write(f"Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        # Write summary of experiment
        papers = set([paper for (paper, _) in all_results["summary_results"].keys()])
        models = set([model for (_, model) in all_results["summary_results"].keys()])
        
        f.write(f"## Experiment Overview\n\n")
        f.write(f"- Number of papers: {len(papers)}\n")
        f.write(f"- Models evaluated: {', '.join(models)}\n")
        f.write(f"- Tasks: Question Answering and Summarization\n\n")
        
        # Write BERTScore results
        f.write("## BERTScore Results\n\n")
        
        # Summaries
        f.write("### Summaries\n\n")
        f.write("| Paper | Model | Precision | Recall | F1 |\n")
        f.write("|-------|-------|-----------|--------|----|\n")
        
        for (paper, model, task) in all_results["bertscore_results"]:
            if task == "summary":
                scores = all_results["bertscore_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['bertscore_precision']:.4f} | {scores['bertscore_recall']:.4f} | {scores['bertscore_f1']:.4f} |\n")
        
        # QA
        f.write("\n### Question Answering\n\n")
        f.write("| Paper | Model | Precision | Recall | F1 |\n")
        f.write("|-------|-------|-----------|--------|----|\n")
        
        for (paper, model, task) in all_results["bertscore_results"]:
            if task == "qa":
                scores = all_results["bertscore_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['bertscore_precision']:.4f} | {scores['bertscore_recall']:.4f} | {scores['bertscore_f1']:.4f} |\n")
        
        # Write ROUGE results
        f.write("\n## ROUGE Results\n\n")
        
        # Summaries
        f.write("### Summaries\n\n")
        f.write("| Paper | Model | ROUGE-1 | ROUGE-2 | ROUGE-L |\n")
        f.write("|-------|-------|---------|---------|--------|\n")
        
        for (paper, model, task) in all_results["rouge_results"]:
            if task == "summary":
                scores = all_results["rouge_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['rouge1_f']:.4f} | {scores['rouge2_f']:.4f} | {scores['rougeL_f']:.4f} |\n")
        
        # QA
        f.write("\n### Question Answering\n\n")
        f.write("| Paper | Model | ROUGE-1 | ROUGE-2 | ROUGE-L |\n")
        f.write("|-------|-------|---------|---------|--------|\n")
        
        for (paper, model, task) in all_results["rouge_results"]:
            if task == "qa":
                scores = all_results["rouge_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['rouge1_f']:.4f} | {scores['rouge2_f']:.4f} | {scores['rougeL_f']:.4f} |\n")
        
        # RAGAS results
        if all_results["ragas_qa_results"]:
            f.write("\n## RAGAS QA Results\n\n")
            f.write("RAGAS results are available in the CSV files.\n\n")
        
        if all_results["ragas_summary_results"]:
            f.write("\n## RAGAS Summary Results\n\n")
            f.write("RAGAS summary results are available in the CSV files.\n\n")
        
        f.write("\n\n*Note: Full results are available in the CSV files in the output directory.*\n")
    
    logger.info(f"Report generated at {report_path}")

In [18]:
if __name__ == "__main__":
    print("RAG Experiment: QA and Summarization for Multiple Papers and Models")
    print("=" * 80)

    # Check and setup API keys
    openai_key, hf_token = check_and_setup_api_keys()
    
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
        gpu_available = True
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Metal Performance Shaders)")
        gpu_available = True
    else:
        print(f"MPS (Metal Performance Shaders) acceleration: {'Not available'}")
        gpu_available = False
    
    print("\nOptions:")
    print("1. Run with default models")
    print("2. Configure custom models")
    print("3. Run with default models + open LLM as judge")
    
    try:
        choice = input("\nSelect option (1-3): ")
        
        if choice == "1":
            # Default configuration
            model_configs = [
                {
                    "model_name": "Llama3-8B",
                    "model_path": "C:/Users/luqma/AI6130/AI6130_Grp/models/Llama-3.1-8B"  # Update with your path
                },
                {
                    "model_name": "DeepSeek-R1-Distill-Llama-8B",
                    "model_path": "C:/Users/luqma/AI6130/AI6130_Grp/models/DeepSeek-R1-Distill-Llama-8B"  # Update with your path
                }
            ]
            
            print("\nRunning with default models...")
            results = run_experiment(
                model_configs=model_configs,
                papers_dir="./papers",
                reference_dir="./reference",
                output_dir="./results",
                use_gpu=gpu_available,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        elif choice == "2":
            # Custom model configuration
            model_configs = []
            
            print("\nEnter details for Model 1:")
            model1_name = input("Model 1 name (e.g., Llama3-8B): ")
            model1_path = input("Model 1 path (local or HF): ")
            
            model_configs.append({
                "model_name": model1_name,
                "model_path": model1_path
            })
            
            print("\nEnter details for Model 2:")
            model2_name = input("Model 2 name (e.g., TinyLlama-1.1B): ")
            model2_path = input("Model 2 path (local or HF)(default: ./models): "or "./models")
            
            model_configs.append({
                "model_name": model2_name,
                "model_path": model2_path
            })
            
            # Optional third model
            add_third = input("\nAdd a third model? (y/n): ").lower()
            if add_third == 'y':
                print("\nEnter details for Model 3:")
                model3_name = input("Model 3 name: ")
                model3_path = input("Model 2 path (local or HF)(default: ./models): "or "./models")
                
                model_configs.append({
                    "model_name": model3_name,
                    "model_path": model3_path
                })
            
            
            papers_dir = input("\nEnter path to papers directory (default: ./papers): ") or "./papers"
            reference_dir = input("Enter path to reference directory (default: ./reference): ") or "./reference"
            output_dir = input("Enter path to output directory (default: ./experiment_results): ") or "./experiment_results"
            
            print("\nRunning with custom models...")
            results = run_experiment(
                model_configs=model_configs,
                papers_dir=papers_dir,
                reference_dir=reference_dir,
                output_dir=output_dir,
                use_gpu=gpu_available,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        elif choice == "3":
            if not has_hf:
                print("Cannot use Hugging Face API as judge without an API token.")
                print("Please rerun and provide a Hugging Face API token.")
                exit()
                
            # Default models with open LLM as judge
            model_configs = [
                {
                    "model_name": "meta-llama/Llama3-8B",
                    "model_path": "C:/Users/luqma/AI6130/AI6130_Grp/models/Llama-3.1-8B"  # Update with your path
                },
                {
                    "model_name": "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
                    "model_path": "C:/Users/luqma/AI6130/AI6130_Grp/models/DeepSeek-R1-Distill-Llama-8B"  # Update with your path
                }
            ]
            
            open_llm_name = input("\nEnter open LLM model name (default: mistralai/Mistral-7B-v0.1): ") or "mistralai/Mistral-7B-v0.1"
            
            print(f"\nRunning with default models and {open_llm_name} as judge...")
            results = run_experiment(
                model_configs=model_configs,
                papers_dir="./papers",
                reference_dir="./reference",
                output_dir="./experiment_results_open_llm",
                use_gpu=gpu_available,
                use_open_llm=True,
                open_llm_name=open_llm_name,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        else:
            print("Invalid choice.")
            
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

RAG Experiment: QA and Summarization for Multiple Papers and Models
OpenAI API key found in environment variables.
Hugging Face API token found in environment variables.
Using CUDA device: NVIDIA GeForce RTX 4070 Ti

Options:
1. Run with default models
2. Configure custom models
3. Run with default models + open LLM as judge



Select option (1-3):  1


2025-04-10 21:55:17,665 [INFO] Using CUDA device: NVIDIA GeForce RTX 4070 Ti
2025-04-10 21:55:17,665 [INFO] ModelManager initialized with device: cuda
2025-04-10 21:55:17,667 [INFO] Models to load: ['Llama3-8B', 'DeepSeek-R1-Distill-Llama-8B']
2025-04-10 21:55:17,668 [INFO] Loading document: papers\Force.pdf
2025-04-10 21:55:17,668 [INFO] Loading document: papers\Force.pdf
2025-04-10 21:55:17,749 [INFO] Successfully loaded Force.pdf (14631 characters)
2025-04-10 21:55:17,750 [INFO] Loading document: papers\Introduction_to_Dynamics.pdf
2025-04-10 21:55:17,750 [INFO] Loading document: papers\Introduction_to_Dynamics.pdf



Running with default models...
Using device: True


2025-04-10 21:55:17,892 [INFO] Successfully loaded Introduction_to_Dynamics.pdf (15516 characters)
2025-04-10 21:55:17,892 [INFO] Loading document: papers\Newtons_Law.pdf
2025-04-10 21:55:17,892 [INFO] Loading document: papers\Newtons_Law.pdf
2025-04-10 21:55:17,896 [WARNING] Ignoring wrong pointing object 10 0 (offset 0)
2025-04-10 21:55:17,896 [WARNING] Ignoring wrong pointing object 28 0 (offset 0)
2025-04-10 21:55:17,896 [WARNING] Ignoring wrong pointing object 32 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 38 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 44 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 46 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 51 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 56 0 (offset 0)
2025-04-10 21:55:17,898 [WARNING] Ignoring wrong pointing object 61 0 (offset 0)
2025-04-10 21:55:17,898 [WAR

In [ ]:

def example_usage():

    #getAPI key
    openai_key, hf_token = check_and_setup_api_keys()
    # Define model configurations
    model_configs = [
        {
            "model_name": "Llama3-8B",
            "model_path": "./models/Llama-3.1-8B"
        },
        {
            "model_name": "DeepSeek-R1-Distill-Llama-8B",
            "model_path": "./models/DeepSeek-R1-Distill-Llama-8B"
        }
    ]
    
    # Set paths
    papers_dir = "./papers"
    reference_dir = "./reference"
    output_dir = "./experiment_results"
    
    # Run experiment
    results = run_experiment(
        model_configs=model_configs,
        papers_dir=papers_dir,
        reference_dir=reference_dir,
        output_dir=output_dir,
        use_gpu=True,
        use_open_llm=False  # Set to True to use open LLM as judge
        openai_key=openai_key
        hf_token=hf_token
    )
    
    return results

# Example with open LLM as judge
def example_with_open_llm():
    # Define model configurations
    model_configs = [
        {
            "model_name": "Llama3-8B",
            "model_path": "./models/Llama-3.1-8B"
        }
    ]
    
    # Set paths
    papers_dir = "./papers"
    reference_dir = "./reference"
    output_dir = "./experiment_results_open_llm"
    
    # Run experiment with open LLM
    results = run_experiment(
        model_configs=model_configs,
        papers_dir=papers_dir,
        reference_dir=reference_dir,
        output_dir=output_dir,
        use_gpu=True,
        use_open_llm=True,
        open_llm_name="mistralai/Mistral-7B-v0.1"
    )

In [ ]:
#PREV_VERSION
import os
#place your OpenAI API Key here.
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

import sys
import platform
import logging
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Enhanced Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("rag_evaluation.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# System Diagnostics Function
def log_system_diagnostics():
    logger.info("System Diagnostics:")
    logger.info(f"Python Version: {sys.version}")
    logger.info(f"Python Executable: {sys.executable}")
    logger.info(f"Platform: {platform.platform()}")
    logger.info(f"Python Implementation: {platform.python_implementation()}")
    
    libraries_to_check = [
        'transformers', 
        'huggingface_hub', 
        'langchain', 
        'ragas', 
        'pandas', 
        'torch'
    ]
    
    for lib in libraries_to_check:
        try:
            module = __import__(lib)
            logger.info(f"{lib}: {getattr(module, '__version__', 'Version not found')}")
        except ImportError:
            logger.warning(f"{lib}: Not installed")

# Environment Configuration
USE_OPENAI = True
USE_HUGGINGFACE = False

# Define paths you path to the paper directory and lesson plan folders here
paper_dir = r"/Users/hafidzjohari/Desktop/NTU MSAI Courses/24S2/AI6130 - LARGE LANGUAGE MODELS/Projects/RAG Project_1/data/papers"
lesson_plans_dir = r"/Users/hafidzjohari/Desktop/NTU MSAI Courses/24S2/AI6130 - LARGE LANGUAGE MODELS/Projects/RAG Project_1/data/lesson_plans"
lesson_plans_paths = [
    os.path.join(lesson_plans_dir, "DeepSeek-R1-Distill-Llama-8B_2112_13492v1.md"),
    os.path.join(lesson_plans_dir, "Llama3-8B_2112_13492v1.md")
]
model_names = ["DeepSeek-R1-Distill-Llama-8B", "Llama3-8B"]

output_dir = "evaluation_RAGAS"
os.makedirs(output_dir, exist_ok=True)

# [Rest of the functions remain the same as in the original script]

def main():
    # Corrected OpenAI API key check
    if USE_OPENAI:
        # Define the API key as a constant within the function
        OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
        
        if os.environ.get("OPENAI_API_KEY") != OPENAI_API_KEY:
            logger.warning(f"OpenAI API key in environment doesn't match the configured key. Resetting...")
            os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
        
    # Log system diagnostics
    log_system_diagnostics()
    
    # Load paper
    paper_files = [f for f in os.listdir(paper_dir) if f.endswith('.pdf')]
    paper_file = os.path.join(paper_dir, paper_files[0]) if paper_files else None
    
    paper_content = load_paper(paper_file) if paper_file else None
    
    # Load lesson plans
    lesson_plans = load_lesson_plans(lesson_plans_paths)
    
    # Setup RAGAS evaluation
    ragas_metrics, ragas_llm = setup_ragas_evaluation()
    
    # Run RAGAS evaluation if available
    ragas_results = None
    if ragas_metrics and paper_content:
        try:
            from ragas import evaluate
            from datasets import Dataset
            from langchain.text_splitter import RecursiveCharacterTextSplitter
            
            # Create text splitter for chunking
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200
            )
            
            # Split the paper content into chunks
            paper_chunks = text_splitter.split_text(paper_content)
            logger.info(f"Created {len(paper_chunks)} paper chunks for context")
            
            # Prepare for RAGAS evaluation
            evaluation_results = []
            
            for i, lesson_plan in enumerate(lesson_plans):
                if i < len(model_names):
                    model_name = model_names[i]
                    logger.info(f"Running RAGAS evaluation for {model_name}")
                    
                    # Create evaluation data in the correct format
                    eval_data = {
                        "question": ["Create a lesson plan based on the Vision Transformer for Small-Size Datasets paper"],
                        "answer": [lesson_plan],
                        "contexts": [[paper_chunks[0]]],  # Use just the first chunk as an example
                        "reference": [paper_content],  # Using full paper as reference
                        "model": [model_name]  # Add model name to the dataset directly
                    }
                    
                    # Create Dataset object
                    dataset = Dataset.from_dict(eval_data)
                    
                    try:
                        # Run evaluation
                        result = evaluate(
                            dataset=dataset,
                            metrics=ragas_metrics,
                            llm=ragas_llm
                        )
                        
                        # Don't modify the result, but store it as-is
                        evaluation_results.append(result)
                        logger.info(f"RAGAS evaluation completed for {model_name}")
                    except Exception as e:
                        logger.error(f"Error during RAGAS evaluation for {model_name}: {str(e)}")
                        logger.error(f"Error details: {repr(e)}")
            
            # Combine results if we have any
            if evaluation_results:
                # Convert each result to a DataFrame and then concatenate
                dfs = []
                for j, result in enumerate(evaluation_results):
                    # Convert result to DataFrame
                    result_df = result.to_pandas()
                    # Add model name column if not already present
                    if "model" not in result_df.columns:
                        result_df["model"] = model_names[j]
                    dfs.append(result_df)
                    
                # Concatenate all dataframes
                if dfs:
                    ragas_results = pd.concat(dfs)
                    logger.info("RAGAS evaluation successful")
                    
                    # Save RAGAS results to CSV
                    ragas_csv_path = os.path.join(output_dir, "ragas_evaluation_results.csv")
                    ragas_results.to_csv(ragas_csv_path, index=False)
                    logger.info(f"Saved RAGAS results to {ragas_csv_path}")
                else:
                    logger.info("No RAGAS evaluation results were generated")
                    
        except Exception as e:
            logger.error(f"Error in RAGAS evaluation process: {str(e)}")
            ragas_results = None
            
    # Text analysis for lesson plans
    text_analysis_results = [
        analyze_lesson_plan(plan, name) 
        for plan, name in zip(lesson_plans, model_names)
    ]
    
    # Visualization and Reporting
    comparison_data = {}
    for result in text_analysis_results:
        model = result['model']
        for key, value in result.items():
            if key not in ['model', 'top_concepts']:
                if key not in comparison_data:
                    comparison_data[key] = {}
                comparison_data[key][model] = value

    comparison_df = pd.DataFrame(comparison_data).T
    
    # Save comparison to CSV
    csv_path = os.path.join(output_dir, "text_analysis_comparison.csv")
    comparison_df.to_csv(csv_path)
    
    # Create visualizations
    plt.figure(figsize=(15, 10))

    metrics_to_plot = [
        ('word_count', 'Word Count'),
        ('structure_completeness', 'Structure Completeness'),
        ('key_concept_coverage', 'Key Concept Coverage'),
        ('section_count', 'Section Count')
    ]
    
    for i, (metric, title) in enumerate(metrics_to_plot[:4]):
        if metric in comparison_df.index:
            plt.subplot(2, 2, i+1)
            comparison_df.loc[metric].plot(kind='bar')
            plt.title(title)
            plt.xticks(rotation=30)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "text_metrics_comparison.png"))
    
    # Generate report
    report_path = os.path.join(output_dir, "evaluation_report.md")
    with open(report_path, "w") as f:
        f.write("# Lesson Plan Evaluation Report\n\n")
        
        # Add summary
        f.write("## Lesson Plan Comparison\n\n")
        f.write("| Metric | " + " | ".join(model_names) + " |\n")
        f.write("|" + "----|" * (len(model_names) + 1) + "\n")
        
        key_metrics = [
            ("Word Count", "word_count"),
            ("Section Count", "section_count"),
            ("Structure Completeness", "structure_completeness"),
            ("Key Concept Coverage", "key_concept_coverage")
        ]
        
        for metric_name, metric_key in key_metrics:
            values = [str(result[metric_key]) for result in text_analysis_results]
            f.write(f"| {metric_name} | " + " | ".join(values) + " |\n")
    
    print(f"Evaluation complete. Check files in {output_dir}")

def load_paper(paper_file):
    """
    Load paper content from a PDF file.
    Requires PyPDF2 or another PDF parsing library.
    """
    try:
        import PyPDF2
        
        with open(paper_file, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text()
        
        return text
    except ImportError:
        logger.error("PyPDF2 is not installed. Please install it to parse PDFs.")
        return None
    except Exception as e:
        logger.error(f"Error loading paper from {paper_file}: {e}")
        return None

def load_lesson_plans(lesson_plan_paths):
    """
    Load lesson plans from markdown files.
    """
    lesson_plans = []
    for path in lesson_plan_paths:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lesson_plans.append(f.read())
        except Exception as e:
            logger.error(f"Error loading lesson plan from {path}: {e}")
    return lesson_plans

def analyze_lesson_plan(lesson_plan, model_name):
    """
    Perform basic text analysis on a lesson plan.
    """
    # Basic text metrics
    word_count = len(lesson_plan.split())
    section_count = len(re.findall(r'^#+\s', lesson_plan, re.MULTILINE))
    
    # Simple structure completeness check
    structure_indicators = [
        'introduction', 'learning objectives', 'main content', 
        'conclusion', 'assessment', 'resources'
    ]
    structure_completeness = sum(
        1 for indicator in structure_indicators 
        if indicator in lesson_plan.lower()
    ) / len(structure_indicators)
    
    # Key concept coverage
    key_concepts = ['transformer', 'vision', 'dataset', 'machine learning', 'neural network']
    key_concept_coverage = sum(
        1 for concept in key_concepts 
        if concept in lesson_plan.lower()
    ) / len(key_concepts)
    
    return {
        'model': model_name,
        'word_count': word_count,
        'section_count': section_count,
        'structure_completeness': structure_completeness,
        'key_concept_coverage': key_concept_coverage
    }

def setup_ragas_evaluation():
    """
    Setup RAGAS evaluation metrics and LLM.
    """
    try:
        from ragas.metrics import (
            context_precision, 
            context_recall, 
            faithfulness, 
            answer_relevancy
        )
        from langchain_openai import ChatOpenAI
        
        # Initialize metrics
        ragas_metrics = [
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy
        ]
        
        # Initialize LLM (using OpenAI's GPT model)
        ragas_llm = ChatOpenAI(model_name="gpt-3.5-turbo")
        
        return ragas_metrics, ragas_llm
    
    except ImportError as e:
        logger.error(f"Error setting up RAGAS evaluation: {e}")
        return None, None

if __name__ == "__main__":
    main()

In [ ]:
#OLD_VERSION_DONT_RUN
def load_document(file_path):
    logger.info(f"Loading document: {file_path}")
    try:
        if file_path.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
            return loader.load()
        elif file_path.endswith(('.txt', '.md')):
            loader = TextLoader(file_path)
            return loader.load()
        else:
            logger.error(f"Unsupported file type: {file_path}")
            return []
    except Exception as e:
        logger.error(f"Error loading file {file_path}: {str(e)}")
        return []

# Function to extract title from document content
def extract_title(content):
    # Try to find a title in the first few lines
    lines = content.split('\n')
    for line in lines[:10]:
        if line.strip() and len(line.strip()) < 100:  # Reasonable title length
            return line.strip()
    
    # If no clear title, use filename
    return "Academic Paper"

# Function to generate lesson plan from paper content
def generate_lesson_plan(model, tokenizer, paper_content, paper_filename, device, seed=None, model_name=""):
    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Extract title from content to use as topic
    paper_title = extract_title(paper_content)
    topic = f"{paper_title} (from {os.path.basename(paper_filename)})"
    
    # Limit context to a reasonable length
    context = paper_content[:2000]
    
    # Create the system prompt
    system_prompt = """You are an educational content creator who specializes in creating detailed, practical lesson plans for teachers. You ALWAYS produce original, complete lesson plans based on academic papers. Your lesson plans follow a clear structure and include all necessary components for classroom implementation."""

    # Create user prompt with the paper content as context
    user_prompt = f"""I need you to create a complete, original lesson plan based on this academic paper:

PAPER TITLE: {paper_title}
PAPER CONTENT:
{context}

Please follow these steps to create your lesson plan:

Step 1: Begin by giving your lesson plan a specific, descriptive title based on the paper's topic.

Step 2: List 3-4 specific learning objectives that use measurable action verbs (explain, analyze, create, etc.) and clearly state what students will be able to do after the lesson.

Step 3: Define 4-5 key concepts from the paper, including brief explanations of their importance and relevance.

Step 4: Design a 50-60 minute teaching activity broken into:
   - Introduction (10 min): How you'll engage students with the paper's topic
   - Main Activity (30-40 min): Detailed, step-by-step description of how to teach the paper's content
   - Conclusion (10 min): How you'll summarize and reinforce the paper's key points

Step 5: Describe both formative assessment (during the lesson) and summative assessment (after the lesson) methods that measure understanding of the paper's concepts.

Step 6: Provide differentiation strategies for different learning needs and abilities.

Write the complete lesson plan in a structured format with clear section headings. Use concrete, specific language throughout - no placeholders or generic content."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
I'll create a complete, original lesson plan based on this academic paper.

"""
    
    # Log the prompt
    logger.info(f"Generating lesson plan for paper: {os.path.basename(paper_filename)} using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1800,  # Increased for more comprehensive lesson plans
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    # Clean up the preamble if present
    if "I'll create a complete, original lesson plan" in assistant_response:
        # Find the end of the preamble
        preamble_end = assistant_response.find("\n\n", assistant_response.find("I'll create"))
        if preamble_end != -1:
            assistant_response = assistant_response[preamble_end:].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

# Process response to clean up any formatting issues
def process_response(response_text):
    """Process a response to extract only the actual lesson plan content without any instructions or formatting"""
    if not response_text or len(response_text) < 10:
        return response_text
    
    # First, try to find lesson plan components - these are the strongest indicators of content
    lesson_components = [
        "LEARNING OBJECTIVES", "Learning Objectives", "Learning objectives",
        "KEY CONCEPTS", "Key Concepts", "Key concepts",
        "TEACHING ACTIVITY", "Teaching Activity", "Teaching activity", "Activity",
        "ASSESSMENT", "Assessment"
    ]
    
    # Try to find the first lesson component
    component_positions = []
    for component in lesson_components:
        pos = response_text.find(component)
        if pos > 0:
            component_positions.append((pos, component))
    
    if component_positions:
        # Sort by position to find the earliest component
        component_positions.sort()
        first_component_pos, first_component = component_positions[0]
        
        # Look for a title or introduction before the first component
        preceding_text = response_text[:first_component_pos].strip()
        title_indicators = ["Title:", "Introduction:", "Lesson Plan:", "# ", "## "]
        
        title_pos = -1
        title_indicator = ""
        for indicator in title_indicators:
            pos = preceding_text.find(indicator)
            if pos >= 0 and (title_pos == -1 or pos < title_pos):
                title_pos = pos
                title_indicator = indicator
        
        if title_pos >= 0:
            # We found a title before the first component
            content_start = title_pos
            # Ensure we're not grabbing text from the prompt
            if "You are an expert" in preceding_text[:title_pos]:
                # Find the last newline before the title
                last_nl = preceding_text[:title_pos].rfind("\n\n")
                if last_nl >= 0:
                    content_start = last_nl + 2
            return response_text[content_start:].strip()
        else:
            # Just start from the first component
            return response_text[first_component_pos:].strip()
    
    # If all else fails, return the original text
    return response_text

# Function to evaluate lesson plan with model
def evaluate_lesson_plan(model, tokenizer, paper_content, lesson_plan, device, model_name=""):
    # Create a prompt that asks the model to evaluate the lesson plan based on the paper
    paper_title = extract_title(paper_content)
    
    # Limit content length to fit within context window
    max_paper_len = 500
    max_lesson_len = 1000
    paper_truncated = paper_content[:max_paper_len] if len(paper_content) > max_paper_len else paper_content
    lesson_truncated = lesson_plan[:max_lesson_len] if len(lesson_plan) > max_lesson_len else lesson_plan
    
    # Create evaluation prompt
    eval_prompt = f"""<|system|>
You are an expert education evaluator who specializes in assessing the quality of lesson plans. You evaluate how well a lesson plan captures the key concepts from academic papers and how effectively it would work in a classroom setting.

<|user|>
Please evaluate this lesson plan that was created based on an academic paper. Rate it from 1-10, where 1 is poor and 10 is excellent.

PAPER TITLE: {paper_title}

PAPER CONTENT:
{paper_truncated}

LESSON PLAN:
{lesson_truncated}

Evaluate this lesson plan on a scale of 1-10. Consider:
1. How well it covers the key concepts from the paper
2. The quality of the learning objectives
3. The appropriateness of the teaching activities
4. The effectiveness of the assessment methods

IMPORTANT: Your response MUST begin with a single number between 1 and 10, followed by your explanation.

<|assistant|>
"""
    
    # Tokenize
    inputs = tokenizer(eval_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info(f"Starting evaluation using model: {model_name}")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,  # Short response with just score and explanation
            do_sample=False,     # Deterministic for evaluation
            temperature=0.1,     # Very low temperature for consistency
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    # Extract score and explanation
    result = extract_score_and_explanation(assistant_response)
    logger.info(f"Evaluation complete: {result}")
    return result


In [ ]:
#NOT_BEING_USED

# Helper function to extract evaluation scores
def extract_score_and_explanation(text):
    """Extract the score and explanation from evaluation text."""
    text = text.strip() if text else ""
    
    # First try with basic patterns
    score_pattern = r"^(\d+)"  # Look for a number at the start
    score_match = re.search(score_pattern, text)
    
    if score_match:
        score = score_match.group(1)
        # Extract the explanation (everything after the number)
        explanation = text[len(score):].strip()
        if explanation.startswith(":") or explanation.startswith("-") or explanation.startswith("."):
            explanation = explanation[1:].strip()
        return f"Score: {score} - Explanation: {explanation}"
    
    # Try other patterns if the basic one fails
    detailed_pattern = r"Score:?\s*(\d+)"
    detailed_match = re.search(detailed_pattern, text)
    
    if detailed_match:
        score = detailed_match.group(1)
        explanation_start = text.find("Score:") + len(f"Score: {score}")
        explanation = text[explanation_start:].strip()
        if explanation.startswith(":") or explanation.startswith("-") or explanation.startswith("."):
            explanation = explanation[1:].strip()
        return f"Score: {score} - Explanation: {explanation}"
    
    # Last resort - assign a default score based on sentiment analysis
    from sentence_transformers import SentenceTransformer, util
    
    positive_words = ["excellent", "good", "effective", "well", "thorough", "comprehensive"]
    negative_words = ["poor", "inadequate", "insufficient", "lacks", "missing", "weak"]
    
    # Simple sentiment-based scoring
    text_lower = text.lower()
    positive_count = sum(1 for word in positive_words if word in text_lower)
    negative_count = sum(1 for word in negative_words if word in text_lower)
    
    # Calculate a score based on positive vs negative
    if positive_count + negative_count > 0:
        score = min(10, max(1, int(5 + 5 * (positive_count - negative_count) / (positive_count + negative_count))))
    else:
        score = 5  # Neutral score if no sentiment words found
        
    return f"Score: {score} - Explanation: {text[:100]}..."

# Function for embedding-based evaluation (optional)
def embedding_evaluate_lesson_plan(paper_content, lesson_plan, device="cpu"):
    """Evaluate lesson plan using embedding similarity metrics"""
    if not ADVANCED_EVAL:
        return {"overall_score": 0, "concept_coverage": 0, "structure_quality": 0, "readability": 0}
    
    try:
        # Create embedding model
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
        
        # Process paper content
        paper_paragraphs = [p for p in paper_content.split("\n\n") if len(p.strip()) > 20]
        paper_sentences = []
        for para in paper_paragraphs:
            para_sentences = [s.strip() for s in para.split(".") if len(s.strip()) > 10]
            paper_sentences.extend(para_sentences)
        
        # Process lesson plan
        lesson_paragraphs = [p for p in lesson_plan.split("\n\n") if len(p.strip()) > 20]
        
        # Calculate overall similarity
        paper_embedding = model.encode(" ".join(paper_paragraphs))
        lesson_embedding = model.encode(" ".join(lesson_paragraphs))
        overall_similarity = float(util.cos_sim(paper_embedding, lesson_embedding).item())
        
        # Calculate concept coverage
        if paper_sentences and lesson_paragraphs:
            paper_sentence_embeddings = model.encode(paper_sentences)
            lesson_paragraph_embeddings = model.encode(lesson_paragraphs)
            
            # For each paper sentence, find the max similarity with any lesson paragraph
            max_similarities = []
            for paper_sent_emb in paper_sentence_embeddings:
                similarities = util.cos_sim(paper_sent_emb, lesson_paragraph_embeddings)
                max_sim = float(torch.max(similarities).item())
                max_similarities.append(max_sim)
            
            concept_coverage = sum(max_similarities) / len(max_similarities)
        else:
            concept_coverage = 0.0
        
        # Structure quality - based on presence of key sections
        key_sections = [
            "learning objectives", "key concepts", "teaching activity",
            "introduction", "assessment", "differentiation"
        ]
        
        lesson_lower = lesson_plan.lower()
        section_count = sum(1 for section in key_sections if section in lesson_lower)
        structure_quality = min(1.0, section_count / 5)  # At least 5 sections for full score
        
        # Readability (simple calculation based on sentence length and structure)
        lesson_sentences = []
        for para in lesson_paragraphs:
            para_sentences = [s.strip() for s in para.split(".") if len(s.strip()) > 5]
            lesson_sentences.extend(para_sentences)
            
        avg_sentence_len = sum(len(s.split()) for s in lesson_sentences) / max(1, len(lesson_sentences))
        readability = 1.0 - min(1.0, max(0.0, (avg_sentence_len - 10) / 20))  # Best around 10-15 words
        
        # Combine into overall score (0-10 scale)
        weights = {
            "overall_similarity": 0.4,
            "concept_coverage": 0.3,
            "structure_quality": 0.2,
            "readability": 0.1
        }
        
        combined_score = (
            weights["overall_similarity"] * overall_similarity +
            weights["concept_coverage"] * concept_coverage +
            weights["structure_quality"] * structure_quality +
            weights["readability"] * readability
        ) * 10
        
        return {
            "overall_score": round(combined_score, 2),
            "overall_similarity": round(overall_similarity, 3),
            "concept_coverage": round(concept_coverage, 3),
            "structure_quality": round(structure_quality, 3),
            "readability": round(readability, 3)
        }
        
    except Exception as e:
        logger.error(f"Error in embedding evaluation: {str(e)}")
        return {"overall_score": 0, "error": str(e)}

# Function to visualize evaluation results
def visualize_evaluation_results(evaluation_results, output_dir="data/evaluation_custom_eval"):
    """Create visualizations of evaluation results"""
    if not ADVANCED_EVAL:
        logger.warning("Visualization libraries not available. Install matplotlib and seaborn.")
        return
    
    try:
        os.makedirs(output_dir, exist_ok=True)
        
        # Get model names from the first result
        model_names = []
        for paper_name, result in evaluation_results.items():
            model_names = list(result.keys())
            break
        
        # Prepare data for visualization
        all_model_scores = {model_name: [] for model_name in model_names}
        all_paper_names = []
        
        # Collect scores from evaluation results
        for paper_name, result in evaluation_results.items():
            all_paper_names.append(os.path.basename(paper_name))
            
            for model_name in model_names:
                if model_name in result:
                    model_eval = result[model_name].get("model_evaluation", "Score: 0 - Explanation: None")
                    score_match = re.search(r"Score:\s*(\d+)", model_eval)
                    score = int(score_match.group(1)) if score_match else 0
                    all_model_scores[model_name].append(score)
                else:
                    all_model_scores[model_name].append(0)
        
        # Create bar chart comparing model scores
        plt.figure(figsize=(12, 6))
        bar_width = 0.35
        index = np.arange(len(all_paper_names))
        
        for i, model_name in enumerate(model_names):
            plt.bar(index + i*bar_width, all_model_scores[model_name], bar_width,
                    label=model_name)
        
        plt.xlabel('Papers')
        plt.ylabel('Scores (0-10)')
        plt.title('Model Evaluation Scores by Paper')
        plt.xticks(index + bar_width/2, [name[:15] + "..." for name in all_paper_names], rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/model_comparison_scores.png")
        plt.close()
        
        # Create radar charts for embedding evaluations if available
        for paper_name, result in evaluation_results.items():
            for model_name in model_names:
                if model_name not in result:
                    continue
                    
                emb_eval = result[model_name].get("embedding_evaluation", {})
                if not emb_eval or not isinstance(emb_eval, dict) or "overall_similarity" not in emb_eval:
                    continue
                
                metrics = ["Overall Similarity", "Concept Coverage", "Structure Quality", "Readability"]
                values = [
                    emb_eval.get("overall_similarity", 0),
                    emb_eval.get("concept_coverage", 0),
                    emb_eval.get("structure_quality", 0),
                    emb_eval.get("readability", 0)
                ]
                
                # Add first value at end to complete the circle
                values.append(values[0])
                
                # Create radar chart
                angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
                angles += angles[:1]  # Close the loop
                
                fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
                ax.plot(angles, values, 'o-', linewidth=2)
                ax.fill(angles, values, alpha=0.25)
                ax.set_thetagrids(np.degrees(angles[:-1]), metrics)
                ax.set_ylim(0, 1)
                ax.grid(True)
                
                paper_basename = os.path.basename(paper_name)
                plt.title(f"{model_name} Quality Metrics: {paper_basename[:20]}...")
                plt.tight_layout()
                
                safe_name = "".join(c if c.isalnum() else "_" for c in paper_basename)
                plt.savefig(f"{output_dir}/radar_{model_name}_{safe_name}.png")
                plt.close()
        
        # Create comparison radar chart for each paper
        for paper_name, result in evaluation_results.items():
            # Check if we have embedding evaluations for all models
            has_all_evals = True
            model_emb_values = {}
            
            for model_name in model_names:
                if model_name not in result:
                    has_all_evals = False
                    break
                
                emb_eval = result[model_name].get("embedding_evaluation", {})
                if not emb_eval or not isinstance(emb_eval, dict) or "overall_similarity" not in emb_eval:
                    has_all_evals = False
                    break
                
                model_emb_values[model_name] = [
                    emb_eval.get("overall_similarity", 0),
                    emb_eval.get("concept_coverage", 0),
                    emb_eval.get("structure_quality", 0),
                    emb_eval.get("readability", 0)
                ]
            
            if has_all_evals:
                metrics = ["Overall Similarity", "Concept Coverage", "Structure Quality", "Readability"]
                angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
                angles += angles[:1]  # Close the loop
                
                fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
                
                for model_name in model_names:
                    values = model_emb_values[model_name] + [model_emb_values[model_name][0]]  # Close the loop
                    ax.plot(angles, values, 'o-', linewidth=2, label=model_name)
                    ax.fill(angles, values, alpha=0.1)
                
                ax.set_thetagrids(np.degrees(angles[:-1]), metrics)
                ax.set_ylim(0, 1)
                ax.grid(True)
                
                paper_basename = os.path.basename(paper_name)
                plt.title(f"Model Comparison: {paper_basename[:20]}...")
                plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
                plt.tight_layout()
                
                safe_name = "".join(c if c.isalnum() else "_" for c in paper_basename)
                plt.savefig(f"{output_dir}/comparison_radar_{safe_name}.png")
                plt.close()
                
        # Create CSV summary
        summary_rows = []
        for paper_name, result in evaluation_results.items():
            paper_basename = os.path.basename(paper_name)
            
            row = {"Paper": paper_basename}
            
            for model_name in model_names:
                if model_name not in result:
                    continue
                
                # Extract model evaluation score
                model_eval = result[model_name].get("model_evaluation", "Score: 0 - Explanation: None")
                score_match = re.search(r"Score:\s*(\d+)", model_eval)
                model_score = int(score_match.group(1)) if score_match else 0
                
                # Extract explanation if available
                expl_match = re.search(r"Explanation:\s*(.+)", model_eval)
                explanation = expl_match.group(1) if expl_match else "None"
                
                # Get embedding evaluation scores
                emb_eval = result[model_name].get("embedding_evaluation", {})
                
                row[f"{model_name}_Score"] = model_score
                row[f"{model_name}_Explanation"] = explanation[:100] + "..." if len(explanation) > 100 else explanation
                
                if isinstance(emb_eval, dict) and "overall_score" in emb_eval:
                    row[f"{model_name}_EmbeddingScore"] = emb_eval.get("overall_score", 0)
                    row[f"{model_name}_Similarity"] = emb_eval.get("overall_similarity", 0)
                    row[f"{model_name}_ConceptCoverage"] = emb_eval.get("concept_coverage", 0)
                    row[f"{model_name}_Structure"] = emb_eval.get("structure_quality", 0)
                    row[f"{model_name}_Readability"] = emb_eval.get("readability", 0)
            
            summary_rows.append(row)
        
        # Create DataFrame and save to CSV
        if summary_rows:
            summary_df = pd.DataFrame(summary_rows)
            summary_csv = os.path.join(output_dir, "evaluation_summary.csv")
            summary_df.to_csv(summary_csv, index=False)
            logger.info(f"Evaluation summary saved to {summary_csv}")
        
        logger.info(f"Visualizations saved to {output_dir}")
    except Exception as e:
        logger.error(f"Error creating visualizations: {str(e)}")

# Function to compare the two models' lesson plans
def compare_lesson_plans(lesson_plan1, lesson_plan2, model_names, paper_content=None, device="cpu"):
    """
    Compare two lesson plans and analyze their similarities and differences.
    Optionally compare both to the original paper content.
    """
    if not ADVANCED_EVAL:
        return {
            "similarity_score": 0,
            "common_sections": [],
            "unique_sections_1": [],
            "unique_sections_2": []
        }
    
    try:
        # Create embedding model
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
        
        # Split plans into sections
        sections1 = [s.strip() for s in lesson_plan1.split("\n\n") if s.strip()]
        sections2 = [s.strip() for s in lesson_plan2.split("\n\n") if s.strip()]
        
        # Calculate overall similarity
        plan1_embedding = model.encode(lesson_plan1)
        plan2_embedding = model.encode(lesson_plan2)
        overall_similarity = float(util.cos_sim(plan1_embedding, plan2_embedding).item())
        
        # Find common and unique sections
        section_embeddings1 = model.encode(sections1)
        section_embeddings2 = model.encode(sections2)
        
        common_sections = []
        unique_sections_1 = []
        unique_sections_2 = []
        
        # Check sections from plan 1
        for i, section1 in enumerate(sections1):
            best_match_score = 0
            best_match_idx = -1
            
            for j, section2 in enumerate(sections2):
                sim_score = float(util.cos_sim(section_embeddings1[i], section_embeddings2[j]).item())
                if sim_score > best_match_score:
                    best_match_score = sim_score
                    best_match_idx = j
            
            if best_match_score > 0.7:  # High similarity threshold
                common_sections.append({
                    "section1": section1,
                    "section2": sections2[best_match_idx],
                    "similarity": best_match_score,
                    "model1": model_names[0],
                    "model2": model_names[1]
                })
            else:
                unique_sections_1.append({
                    "section": section1,
                    "model": model_names[0]
                })
        
        # Find unique sections in plan 2
        for i, section2 in enumerate(sections2):
            if not any(common["section2"] == section2 for common in common_sections):
                unique_sections_2.append({
                    "section": section2,
                    "model": model_names[1]
                })
        
        # Also compare to original paper if provided
        paper_comparison = None
        if paper_content:
            plan1_to_paper = float(util.cos_sim(plan1_embedding, model.encode(paper_content)).item())
            plan2_to_paper = float(util.cos_sim(plan2_embedding, model.encode(paper_content)).item())
            
            paper_comparison = {
                f"{model_names[0]}_to_paper_similarity": plan1_to_paper,
                f"{model_names[1]}_to_paper_similarity": plan2_to_paper,
                "difference": plan2_to_paper - plan1_to_paper,
                "better_match": model_names[1] if plan2_to_paper > plan1_to_paper else model_names[0]
            }
        
        return {
            "similarity_score": overall_similarity,
            "common_sections": common_sections,
            "unique_sections_1": unique_sections_1,
            "unique_sections_2": unique_sections_2,
            "paper_comparison": paper_comparison
        }
        
    except Exception as e:
        logger.error(f"Error in lesson plan comparison: {str(e)}")
        return {"error": str(e)}

# Function to create a detailed analysis report comparing model performances
def create_analysis_report(results, output_dir="data/analysis"):
    """
    Create a comprehensive analysis report comparing the performance of different models.
    
    Args:
        results: Dictionary containing evaluation results for different papers and models
        output_dir: Directory to save the analysis report
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Extract model names
    model_names = []
    for paper_name, result in results.items():
        model_names = list(result.keys())
        model_names = [name for name in model_names if name != "comparison"]
        if len(model_names) >= 2:
            break
    
    if len(model_names) < 2:
        logger.warning("Not enough models to compare for analysis report")
        return
    
    # We'll use the first two models for comparison
    model_1 = model_names[0]
    model_2 = model_names[1]
    
    # Initialize metrics storage
    metrics = {
        "Query Similarity": [],
        "Context Relevance": [],
        "Response Coherence": [],
        "Factual Accuracy": [],
        "Completeness": []
    }
    
    # Collect metrics across all papers
    for paper_name, result in results.items():
        if model_1 in result and model_2 in result:
            # Extract embedding evaluations
            emb_eval_1 = result[model_1].get("embedding_evaluation", {})
            emb_eval_2 = result[model_2].get("embedding_evaluation", {})
            
            # Extract model evaluations
            model_eval_1 = result[model_1].get("model_evaluation", "Score: 0 - Explanation: None")
            model_eval_2 = result[model_2].get("model_evaluation", "Score: 0 - Explanation: None")
            
            # Extract scores
            score_match_1 = re.search(r"Score:\s*(\d+)", model_eval_1)
            score_match_2 = re.search(r"Score:\s*(\d+)", model_eval_2)
            model_score_1 = int(score_match_1.group(1)) / 10 if score_match_1 else 0
            model_score_2 = int(score_match_2.group(1)) / 10 if score_match_2 else 0
            
            # Map metrics from our evaluations to the report metrics
            if isinstance(emb_eval_1, dict) and isinstance(emb_eval_2, dict):
                # Query Similarity - using overall similarity as proxy
                metrics["Query Similarity"].append((
                    emb_eval_1.get("overall_similarity", 0),
                    emb_eval_2.get("overall_similarity", 0)
                ))
                
                # Context Relevance - using concept coverage as proxy
                metrics["Context Relevance"].append((
                    emb_eval_1.get("concept_coverage", 0),
                    emb_eval_2.get("concept_coverage", 0)
                ))
                
                # Response Coherence - using readability as proxy
                metrics["Response Coherence"].append((
                    emb_eval_1.get("readability", 0),
                    emb_eval_2.get("readability", 0)
                ))
                
                # Factual Accuracy - using structure quality as proxy
                metrics["Factual Accuracy"].append((
                    emb_eval_1.get("structure_quality", 0),
                    emb_eval_2.get("structure_quality", 0)
                ))
                
                # Completeness - using model evaluation score as proxy
                metrics["Completeness"].append((model_score_1, model_score_2))
    
    # Calculate averages for each metric
    avg_metrics = {}
    for metric_name, values in metrics.items():
        if values:
            avg_model_1 = sum(v[0] for v in values) / len(values)
            avg_model_2 = sum(v[1] for v in values) / len(values)
            diff = avg_model_2 - avg_model_1
            avg_metrics[metric_name] = (avg_model_1, avg_model_2, diff)
    
    # Create markdown report
    report = f"# RAG System Performance Analysis\n\n"
    report += f"## Model Comparison: {model_1} vs {model_2}\n\n"
    
    # Create metrics table
    report += "| Metric             |   {} |   {} |   Difference |\n".format(model_1, model_2)
    report += "|:-------------------|----------------:|-------------------:|-------------:|\n"
    
    for metric_name, (avg_1, avg_2, diff) in avg_metrics.items():
        report += f"| {metric_name:<18} | {avg_1:>15.6f} | {avg_2:>18.6f} | {diff:>12.6f} |\n"
    
    report += "\n## Key Findings\n\n"
    
    # Determine model strengths based on metrics
    model_1_strengths = []
    model_2_strengths = []
    
    for metric_name, (avg_1, avg_2, diff) in avg_metrics.items():
        if diff < 0:  # Model 1 is better
            model_1_strengths.append(f"- **{metric_name}**: {avg_1:.4f} vs {avg_2:.4f} (difference: {diff:.4f})")
        elif diff > 0:  # Model 2 is better
            model_2_strengths.append(f"- **{metric_name}**: {avg_2:.4f} vs {avg_1:.4f} (difference: {diff:.4f})")
    
    # Add strengths to report
    report += f"### {model_1} Strengths\n\n"
    if model_1_strengths:
        report += "\n".join(model_1_strengths)
    else:
        report += "- No clear advantages identified\n"
    
    report += f"\n\n### {model_2} Strengths\n\n"
    if model_2_strengths:
        report += "\n".join(model_2_strengths)
    else:
        report += "- No clear advantages identified\n"
    
    # Add recommendations
    report += "\n\n## Recommendations\n\n"
    report += "Based on the analysis, here are some recommendations:\n\n"
    
    # Determine overall better model
    model_1_better_count = len(model_1_strengths)
    model_2_better_count = len(model_2_strengths)
    
    if model_1_better_count > model_2_better_count:
        report += f"- The **{model_1}** performs better overall and should be preferred for most use cases\n"
    elif model_2_better_count > model_1_better_count:
        report += f"- The **{model_2}** performs better overall and should be preferred for most use cases\n"
    else:
        report += "- Both models have similar overall performance; choose based on specific use case requirements\n"
    
    report += "- Experiment with different context window sizes to find the optimal balance\n"
    report += f"- Consider ensemble approaches combining strengths of both {model_1} and {model_2}\n"
    
    # Save the report
    report_path = os.path.join(output_dir, "model_analysis_report.md")
    with open(report_path, "w") as f:
        f.write(report)
    
    logger.info(f"Analysis report saved to {report_path}")
    
    # Generate visual comparison chart
    try:
        plt.figure(figsize=(10, 6))
        metrics_list = list(avg_metrics.keys())
        model_1_scores = [avg_metrics[m][0] for m in metrics_list]
        model_2_scores = [avg_metrics[m][1] for m in metrics_list]
        
        x = np.arange(len(metrics_list))
        width = 0.35
        
        plt.bar(x - width/2, model_1_scores, width, label=model_1)
        plt.bar(x + width/2, model_2_scores, width, label=model_2)
        
        plt.xlabel('Metrics')
        plt.ylabel('Scores')
        plt.title(f'Model Comparison: {model_1} vs {model_2}')
        plt.xticks(x, metrics_list, rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        
        chart_path = os.path.join(output_dir, "model_comparison_chart.png")
        plt.savefig(chart_path)
        plt.close()
        
        logger.info(f"Comparison chart saved to {chart_path}")
    except Exception as e:
        logger.error(f"Error creating comparison chart: {str(e)}")
    
    return report_path

# Main function with multi-model support
# Main function with enhanced analysis reporting
def run_paper_based_lesson_generator_with_multiple_models(model_configs, use_mps=True):
    """
    Run the paper-based lesson generator with multiple models
    
    Args:
        model_configs: List of dictionaries with model_path and model_name keys
        use_mps: Whether to use MPS for acceleration
    """
    # Check if MPS is available
    mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    device = "mps" if (use_mps and mps_available) else "cpu"
    logger.info(f"Setting up with device: {device}")
    
    # Set directories
    papers_dir = "data/papers"
    lesson_plans_dir = "data/lesson_plans"
    evaluation_dir = "data/evaluation"
    comparison_dir = "data/comparison"
    analysis_dir = "data/analysis"
    
    # Create output directories if they don't exist
    os.makedirs(lesson_plans_dir, exist_ok=True)
    os.makedirs(evaluation_dir, exist_ok=True)
    os.makedirs(comparison_dir, exist_ok=True)
    os.makedirs(analysis_dir, exist_ok=True)
    
    # Initialize the model manager
    model_manager = ModelManager(model_configs, use_mps=use_mps)
    model_names = model_manager.get_model_names()
    
    # Store the results for each paper and model
    results = {}
    
    try:
        # Check if papers directory exists
        if not os.path.exists(papers_dir):
            logger.error(f"Papers directory does not exist: {papers_dir}")
            return
        
        # Get list of papers
        paper_files = [f for f in os.listdir(papers_dir) if not f.startswith('.') and not os.path.isdir(os.path.join(papers_dir, f))]
        
        # Process each paper with each model
        for paper_idx, filename in enumerate(paper_files):
            file_path = os.path.join(papers_dir, filename)
            
            # Load the document
            doc_sections = load_document(file_path)
            if not doc_sections:
                logger.warning(f"No content loaded from {filename}, skipping")
                continue
                
            # Combine all sections into one text
            paper_content = "\n\n".join([section.page_content for section in doc_sections])
            
            # Extract a base name for output files
            base_name = os.path.splitext(filename)[0]
            sanitized_name = ''.join(c if c.isalnum() or c in ['-', '_'] else '_' for c in base_name)
            
            # Create result storage for this paper
            results[file_path] = {}
            
            # Process with each model
            for model_idx, model_name in enumerate(model_names):
                logger.info(f"Processing paper {paper_idx+1}/{len(paper_files)}: {filename} with model: {model_name}")
                
                try:
                    # Load the model
                    model, tokenizer = model_manager.load_model(model_name)
                    
                    # Generate lesson plan
                    lesson_plan = generate_lesson_plan(
                        model, tokenizer, paper_content, file_path, device, 
                        seed=hash(filename) % 10000,  # Use filename hash for seed
                        model_name=model_name
                    )
                    
                    # Process the response to clean it up
                    processed_lesson_plan = process_response(lesson_plan)
                    if len(processed_lesson_plan) > 50:  # Only use processed version if it's substantial
                        lesson_plan = processed_lesson_plan
                    
                    # Save the lesson plan
                    output_file = os.path.join(lesson_plans_dir, f"{model_name}_{sanitized_name}.md")
                    with open(output_file, "w") as f:
                        f.write(lesson_plan)
                    
                    logger.info(f"Saved lesson plan to {output_file}")
                    
                    # Print preview
                    preview_length = min(300, len(lesson_plan))
                    logger.info(f"Preview:\n{lesson_plan[:preview_length]}...\n")
                    
                    # Evaluate the lesson plan
                    logger.info(f"Evaluating lesson plan for: {filename} with model: {model_name}")
                    
                    # Model-based evaluation
                    model_evaluation = evaluate_lesson_plan(
                        model, tokenizer, paper_content, lesson_plan, device, model_name
                    )
                    
                    # Embedding-based evaluation
                    embedding_evaluation = embedding_evaluate_lesson_plan(paper_content, lesson_plan, device=device)
                    
                    # Store the results
                    results[file_path][model_name] = {
                        "lesson_plan": lesson_plan,
                        "model_evaluation": model_evaluation,
                        "embedding_evaluation": embedding_evaluation
                    }
                    
                    # Save individual evaluation to file
                    eval_output_file = os.path.join(evaluation_dir, f"{model_name}_{sanitized_name}_eval.txt")
                    with open(eval_output_file, "w") as f:
                        f.write(f"MODEL EVALUATION:\n{model_evaluation}\n\n")
                        f.write(f"EMBEDDING EVALUATION:\n")
                        for key, value in embedding_evaluation.items():
                            f.write(f"{key}: {value}\n")
                    
                    logger.info(f"Evaluation complete and saved to {eval_output_file}")
                    
                    # Unload model to free memory
                    model_manager.unload_model(model_name)
                    clear_memory()
                    
                except Exception as e:
                    logger.error(f"Error processing {filename} with model {model_name}: {str(e)}", exc_info=True)
                    # Store the error in results
                    results[file_path][model_name] = {
                        "error": str(e)
                    }
            
            # Compare lesson plans if we have results from both models
            if len(results[file_path]) >= 2 and all("lesson_plan" in results[file_path][model_name] for model_name in model_names[:2]):
                try:
                    logger.info(f"Comparing lesson plans for {filename}")
                    
                    # Get lesson plans for the first two models
                    lesson_plan1 = results[file_path][model_names[0]]["lesson_plan"]
                    lesson_plan2 = results[file_path][model_names[1]]["lesson_plan"]
                    
                    # Compare them
                    comparison = compare_lesson_plans(
                        lesson_plan1, lesson_plan2, model_names[:2], paper_content, device
                    )
                    
                    # Save comparison to file
                    comparison_file = os.path.join(comparison_dir, f"{sanitized_name}_comparison.txt")
                    
                    with open(comparison_file, "w") as f:
                        f.write(f"COMPARISON OF LESSON PLANS FOR: {filename}\n")
                        f.write(f"Model 1: {model_names[0]}\n")
                        f.write(f"Model 2: {model_names[1]}\n\n")
                        
                        f.write(f"Overall Similarity: {comparison['similarity_score']:.4f}\n\n")
                        
                        if comparison.get('paper_comparison'):
                            pc = comparison['paper_comparison']
                            f.write("PAPER SIMILARITY:\n")
                            f.write(f"{model_names[0]} to paper: {pc[f'{model_names[0]}_to_paper_similarity']:.4f}\n")
                            f.write(f"{model_names[1]} to paper: {pc[f'{model_names[1]}_to_paper_similarity']:.4f}\n")
                            f.write(f"Difference: {pc['difference']:.4f}\n")
                            f.write(f"Better match to paper: {pc['better_match']}\n\n")
                        
                        f.write("COMMON SECTIONS:\n")
                        for i, common in enumerate(comparison['common_sections']):
                            f.write(f"[{i+1}] Similarity: {common['similarity']:.4f}\n")
                            f.write(f"{model_names[0]}: {common['section1'][:150]}...\n")
                            f.write(f"{model_names[1]}: {common['section2'][:150]}...\n\n")
                        
                        f.write(f"UNIQUE SECTIONS IN {model_names[0]}:\n")
                        for i, unique in enumerate(comparison['unique_sections_1']):
                            f.write(f"[{i+1}] {unique['section'][:200]}...\n\n")
                        
                        f.write(f"UNIQUE SECTIONS IN {model_names[1]}:\n")
                        for i, unique in enumerate(comparison['unique_sections_2']):
                            f.write(f"[{i+1}] {unique['section'][:200]}...\n\n")
                    
                    logger.info(f"Comparison saved to {comparison_file}")
                    
                    # Store the comparison in results
                    results[file_path]["comparison"] = comparison
                    
                except Exception as e:
                    logger.error(f"Error comparing lesson plans for {filename}: {str(e)}", exc_info=True)
            
            # Clean up after processing each paper
            clear_memory()
        
        # Create visualizations of the results
        visualize_evaluation_results(results, evaluation_dir)
        
        # Generate enhanced analysis report
        analysis_report_path = create_analysis_report(results, analysis_dir)
        if analysis_report_path:
            logger.info(f"Detailed analysis report generated at {analysis_report_path}")
        
        logger.info("Processing complete!")
        return results
        
    except Exception as e:
        logger.error(f"Error in main processing: {str(e)}", exc_info=True)
        return results

# Example usage function
def run_example():
    # Define model configurations
    model_configs = [
        {
            "model_name": "Llama3-8B",
            "model_path": "/Users/hafidzjohari/Llama-3.1-8B-v2"
        },
        {
            "model_name": "DeepSeek-R1-Distill-Llama-8B",
            "model_path": "/Users/hafidzjohari/DeepSeek-R1-Distill-Llama-8B"
        }
    ]
    
    # Run with multiple models
    results = run_paper_based_lesson_generator_with_multiple_models(model_configs, use_mps=True)
    
    return results

# Enhanced main execution
if __name__ == "__main__":
    print("Enhanced Paper-Based Lesson Plan Generator with Multi-Model Support")
    print("=" * 70)
    
    # Check for MPS availability
    mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    print(f"MPS (Metal Performance Shaders) acceleration: {'Available' if mps_available else 'Not available'}")
    
    print("\nOptions:")
    print("1. Run with default models (Llama-3.1-8B and DeepSeek-R1-Distill-Llama-8B)")
    print("2. Configure custom models")
    print("3. Run evaluation on existing lesson plans")
    
    try:
        choice = input("\nSelect option (1-3): ")
        
        if choice == "1":
            # Default configuration
            model_configs = [
                {
                    "model_name": "Llama3-8B",
                    "model_path": "/Users/hafidzjohari/Llama-3.1-8B-v2"
                },
                {
                    "model_name": "DeepSeek-R1-Distill-Llama-8B",
                    "model_path": "/Users/hafidzjohari/DeepSeek-R1-Distill-Llama-8B"
                }
            ]
            
            print("\nRunning with default models...")
            results = run_paper_based_lesson_generator_with_multiple_models(model_configs, use_mps=True)
            
        elif choice == "2":
            # Custom model configuration
            model_configs = []
            
            print("\nEnter details for Model 1:")
            model1_name = input("Model 1 name (e.g., Llama3-8B): ")
            model1_path = input("Model 1 path (local or HF): ")
            
            model_configs.append({
                "model_name": model1_name,
                "model_path": model1_path
            })
            
            print("\nEnter details for Model 2:")
            model2_name = input("Model 2 name (e.g., TinyLlama-1.1B): ")
            model2_path = input("Model 2 path (local or HF): ")
            
            model_configs.append({
                "model_name": model2_name,
                "model_path": model2_path
            })
            
            # Optional third model
            add_third = input("\nAdd a third model? (y/n): ").lower()
            if add_third == 'y':
                print("\nEnter details for Model 3:")
                model3_name = input("Model 3 name: ")
                model3_path = input("Model 3 path (local or HF): ")
                
                model_configs.append({
                    "model_name": model3_name,
                    "model_path": model3_path
                })
            
            use_mps = input("\nUse MPS acceleration if available? (y/n): ").lower() == 'y'
            
            print("\nRunning with custom models...")
            results = run_paper_based_lesson_generator_with_multiple_models(model_configs, use_mps=use_mps)
            
        elif choice == "3":
            # Evaluation on existing lesson plans
            print("\nEvaluating existing lesson plans...")
            
            lesson_plans_dir = "data/lesson_plans"
            if not os.path.exists(lesson_plans_dir):
                print(f"Error: Lesson plans directory not found: {lesson_plans_dir}")
            else:
                # Get list of lesson plan files
                lesson_files = os.listdir(lesson_plans_dir)
                
                if not lesson_files:
                    print("No lesson plan files found.")
                else:
                    print("Available lesson plans:")
                    for i, file in enumerate(lesson_files):
                        print(f"{i+1}. {file}")
                    
                    # Get files to compare
                    file1_idx = int(input("\nSelect first lesson plan (number): ")) - 1
                    file2_idx = int(input("Select second lesson plan (number): ")) - 1
                    
                    if 0 <= file1_idx < len(lesson_files) and 0 <= file2_idx < len(lesson_files):
                        file1_path = os.path.join(lesson_plans_dir, lesson_files[file1_idx])
                        file2_path = os.path.join(lesson_plans_dir, lesson_files[file2_idx])
                        
                        # Get model names
                        model1_name = input("\nEnter name for first model: ")
                        model2_name = input("Enter name for second model: ")
                        
                        # Optional paper path
                        paper_path = input("\nEnter path to original paper (or leave empty): ")
                        paper_content = None
                        if paper_path and os.path.exists(paper_path):
                            doc_sections = load_document(paper_path)
                            if doc_sections:
                                paper_content = "\n\n".join([section.page_content for section in doc_sections])
                        
                        # Read lesson plans
                        with open(file1_path, "r") as f:
                            lesson_plan1 = f.read()
                        
                        with open(file2_path, "r") as f:
                            lesson_plan2 = f.read()
                        
                        # Compare
                        device = "mps" if mps_available else "cpu"
                        comparison = compare_lesson_plans(
                            lesson_plan1, lesson_plan2, [model1_name, model2_name], 
                            paper_content, device
                        )
                        
                        # Save comparison
                        comparison_dir = "data/comparison"
                        os.makedirs(comparison_dir, exist_ok=True)
                        comparison_file = os.path.join(comparison_dir, "manual_comparison.txt")
                        
                        with open(comparison_file, "w") as f:
                            f.write(f"COMPARISON OF LESSON PLANS\n")
                            f.write(f"Model 1: {model1_name} - {os.path.basename(file1_path)}\n")
                            f.write(f"Model 2: {model2_name} - {os.path.basename(file2_path)}\n\n")
                            
                            f.write(f"Overall Similarity: {comparison['similarity_score']:.4f}\n\n")
                            
                            if comparison.get('paper_comparison'):
                                pc = comparison['paper_comparison']
                                f.write("PAPER SIMILARITY:\n")
                                f.write(f"{model1_name} to paper: {pc[f'{model1_name}_to_paper_similarity']:.4f}\n")
                                f.write(f"{model2_name} to paper: {pc[f'{model2_name}_to_paper_similarity']:.4f}\n")
                                f.write(f"Difference: {pc['difference']:.4f}\n")
                                f.write(f"Better match to paper: {pc['better_match']}\n\n")
                            
                            f.write("COMMON SECTIONS:\n")
                            for i, common in enumerate(comparison['common_sections']):
                                f.write(f"[{i+1}] Similarity: {common['similarity']:.4f}\n")
                                f.write(f"{model1_name}: {common['section1'][:150]}...\n")
                                f.write(f"{model2_name}: {common['section2'][:150]}...\n\n")
                            
                            f.write(f"UNIQUE SECTIONS IN {model1_name}:\n")
                            for i, unique in enumerate(comparison['unique_sections_1']):
                                f.write(f"[{i+1}] {unique['section'][:200]}...\n\n")
                            
                            f.write(f"UNIQUE SECTIONS IN {model2_name}:\n")
                            for i, unique in enumerate(comparison['unique_sections_2']):
                                f.write(f"[{i+1}] {unique['section'][:200]}...\n\n")
                        
                        print(f"\nComparison saved to {comparison_file}")
                        
                        # Print summary
                        print(f"\nSummary of comparison:")
                        print(f"Overall similarity: {comparison['similarity_score']:.4f}")
                        print(f"Common sections: {len(comparison['common_sections'])}")
                        print(f"Unique to {model1_name}: {len(comparison['unique_sections_1'])}")
                        print(f"Unique to {model2_name}: {len(comparison['unique_sections_2'])}")
                        
                        if comparison.get('paper_comparison'):
                            pc = comparison['paper_comparison']
                            print(f"\nBetter match to paper: {pc['better_match']}")
                    else:
                        print("Invalid file selection.")
        else:
            print("Invalid choice.")
            
    except Exception as e:
        print(f"Error: {str(e)}")